In [9]:
!pip install librosa tensorflow numpy matplotlib tqdm

In [26]:
import librosa
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import random
from tqdm import tqdm

# Training Data
Can be found here: https://www.kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset

The data is not stored in the github for file size reasons. The dataset needs to be downloaded for code to function properly.

In [31]:
# Only using 2-sec folder of the data since we are only examining short form audio we 
FAKE_TRAINING_DATA = "archive/for-2sec/for-2seconds/training/fake"
REAL_TRAINING_DATA = "archive/for-2sec/for-2seconds/training/real"

FAKE_TESTING_DATA = "archive/for-2sec/for-2seconds/testing/fake"
REAL_TESTING_DATA = "archive/for-2sec/for-2seconds/testing/real"

FAKE_VALIDATION_DATA = "archive/for-2sec/for-2seconds/validation/fake/"
REAL_VALIDATION_DATA = "archive/for-2sec/for-2seconds/validation/real/"

print(len(os.listdir(FAKE_TRAINING_DATA)))
print(len(os.listdir(REAL_TRAINING_DATA)))

6978
6978


In [32]:
def process_for_2sec_file(file_path):
    # 'duration=1.0' tells librosa to only load the first second  of the 2-second file in the for-2sec folder.
    y, sr = librosa.load(file_path, sr=22050, duration=1.0)
    
    # in case a file is slightly under 1 second, fix the length
    y = librosa.util.fix_length(y, size=sr)
    
    # Create melspectrogram
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    log_S = librosa.power_to_db(S)
    delta_S = librosa.feature.delta(log_S)
    log_S = np.stack([log_S, delta_S], axis=-1)
    return log_S


def iterate_folder(data_folder, max_files=None):
    data_list = []
    
    # Get all .wav files first
    files = [f for f in os.listdir(data_folder) if f.endswith(".wav")]

    random.shuffle(files)
    # If a limit is set, take only that many
    if max_files is not None:
        files = files[:max_files]

    for filename in tqdm(files, desc=f"Processing {os.path.basename(data_folder)}", unit="file"):
        file_path = os.path.join(data_folder, filename)
        result = process_for_2sec_file(file_path)
        data_list.append(result)
        
    return data_list

# Get list of real and fake training data
fake_train_data, real_train_data = iterate_folder(FAKE_TRAINING_DATA, max_files=6978), iterate_folder(REAL_TRAINING_DATA, max_files=6978)

# Get list of real and fake test data
fake_test_data, real_test_data = iterate_folder(FAKE_TESTING_DATA, max_files=1000), iterate_folder(REAL_TESTING_DATA, max_files=1000)

# Get list of real and fake validation data
fake_valid_data, real_valid_data = iterate_folder(FAKE_VALIDATION_DATA), iterate_folder(REAL_VALIDATION_DATA)

Processing : 100%|██████████| 1413/1413 [00:46<00:00, 30.54file/s]


# Create Answer Key
We need to assign an answer key to the training data. In this case, 0's will indicate real audio files, 1's will indicate fake audio files

In [33]:
def convert_to_numpy(real, fake):
    # Convert lists to numpy arrays
    X_real = np.array(real)
    X_fake = np.array(fake)

    # Create labels (0s for real, 1s for fake)
    y_real = np.zeros(len(X_real))
    y_fake = np.ones(len(X_fake))

    # Stack them together
    X = np.concatenate((X_real, X_fake), axis=0)
    y = np.concatenate((y_real, y_fake), axis=0)

    return X, y

X_train, y_train = convert_to_numpy(real_train_data, fake_train_data)
X_test, y_test = convert_to_numpy(real_test_data, fake_test_data)
X_val, y_val = convert_to_numpy(real_valid_data, fake_valid_data)


print(X_train.shape)

(13956, 128, 44, 2)


# Making the Spectrogram 4D
CNNs need to have a 4D format, therefore we need to add an extra parameter to each spectrogram

In this case the four dimensions are:
1. Number of 1-Second iterations
2. Frequency Resolution (Mel Bins)
3. Time Resolution (1 second divded by hop length)
4. Color Channel Intensity ()

# Normalize the Data
The numbers are too large to be efficiently computed by a CNN, we need to normalize the data now

In [34]:
# # 1. Calculate the 'Master' Min and Max from Training ONLY
# train_min = X_train.min()
# train_max = X_train.max()

# def apply_master_normalization(data, master_min, master_max):
#     return (data - master_min) / (master_max - master_min)

# X_train = apply_master_normalization(X_train, train_min, train_max)
# X_val   = apply_master_normalization(X_val, train_min, train_max)
# X_test  = apply_master_normalization(X_test, train_min, train_max)

def normalize_global_per_channel(X_train, X_val, X_test):
    # 1. Find min/max for Log-Mel (Channel 0) across the whole training set
    mel_min = X_train[..., 0].min()
    mel_max = X_train[..., 0].max()
    
    # 2. Find min/max for Delta (Channel 1) across the whole training set
    delta_min = X_train[..., 1].min()
    delta_max = X_train[..., 1].max()
    
    # 3. Apply these master values to all sets
    def apply_norm(data):
        data[..., 0] = (data[..., 0] - mel_min) / (mel_max - mel_min + 1e-8)
        data[..., 1] = (data[..., 1] - delta_min) / (delta_max - delta_min + 1e-8)
        return data

    return apply_norm(X_train), apply_norm(X_val), apply_norm(X_test)

# Overwrite your datasets using the new function
X_train, X_val, X_test = normalize_global_per_channel(X_train, X_val, X_test)


# Save the data so other notebooks can access it

In [35]:
from sklearn.utils import shuffle

# Shuffle Training Data
X_train, y_train = shuffle(X_train, y_train, random_state=42)

# Shuffle Validation Data
X_val, y_val = shuffle(X_val, y_val, random_state=42)

# Shuffle Testing Data
X_test, y_test = shuffle(X_test, y_test, random_state=42)

# Save X and y into a single compressed file
np.savez_compressed('processed_data.npz', 
                    x_train=X_train,
                    y_train=y_train,
                    x_val=X_val,
                    y_val=y_val,
                    x_test=X_test,
                    y_test=y_test)
print("Successfully saved to 'processed_data.npz'")



Successfully saved to 'processed_data.npz'


In [93]:
print(X_train.shape)

(12151, 128, 44, 2)
